In [1]:
%pip install sqlalchemy "psycopg[binary]"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\Acer\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import sqlalchemy
import psycopg

print("SQLAlchemy:", sqlalchemy.__version__)
print("Psycopg:", psycopg.__version__)

SQLAlchemy: 2.1.5
Psycopg: 3.3.6


In [4]:
%pip install jupysql

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
   ---------------------------------------- 0.0/892.2 kB ? eta -:--:--
   ---------------------------------------- 892.2/892.2 kB 8.1 MB/s  0:00:00
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
   ---------------------------------------- 0.0/783.7 kB ? eta -:--:--
   ---------------------------------------- 783.7/783.7 kB 16.8 MB/s  0:00:00
   ---------------------------------------- 0.0/597.8 kB ? eta -:--:--
   ---------------------------------------- 597.8/597.8 kB 10.8 MB/s  0:00:00

   ----------------------------------------  0/11 [ipython-genutils]
  Attempting uninstall: wcwidth
   ----------------------------------------  0/11 [ipython-genutils]
    Found existing installation: wcwidth 0.2.14
   ----------------------------------------  0/11 [ipython-genutils]
   --- ------------------------------------  1/11 [wcwidth]
    Uninstalling wcwidth-0.2.14:
   --- ------------------------------------  1/11 [wcwidth]

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ftfy 6.1.3 requires wcwidth<0.3.0,>=0.2.12, but you have wcwidth 0.9.2 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\Acer\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
%load_ext sql

In [7]:
from getpass import getpass
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

password = getpass("Enter PostgreSQL password: ")

db_url = URL.create(
    "postgresql+psycopg",
    username="postgres",
    password=password,
    host="localhost",
    port=5432,
    database="postgres"
)

engine = create_engine(db_url)

with engine.connect() as connection:
    result = connection.execute(text("SELECT current_database();"))
    print("Connected to:", result.scalar())

Connected to: postgres


In [10]:
%load_ext sql
%sql engine

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [11]:
%config SqlMagic.style = 'DEFAULT'

In [13]:
import pandas as pd
from sqlalchemy import text

with engine.connect() as conn:
    df = pd.read_sql(
        text("""
            SELECT
                current_database() AS database_name,
                current_user AS username;
        """),
        conn
    )

display(df)

,database_name,username
0,postgres,postgres


In [14]:
from sqlalchemy import text

with engine.connect().execution_options(
    isolation_level="AUTOCOMMIT"
) as conn:
    exists = conn.execute(
        text("SELECT 1 FROM pg_database WHERE datname = :name"),
        {"name": "aviation_covid_analysis"}
    ).scalar()

    if not exists:
        conn.execute(text("CREATE DATABASE aviation_covid_analysis"))
        print("Database created successfully!")
    else:
        print("Database already exists!")

Database created successfully!


In [15]:
from sqlalchemy import create_engine

project_url = db_url.set(database="aviation_covid_analysis")
project_engine = create_engine(project_url)

with project_engine.connect() as conn:
    database = conn.execute(
        text("SELECT current_database()")
    ).scalar()

    print("Connected to:", database)

Connected to: aviation_covid_analysis


In [16]:
import pandas as pd
from pathlib import Path

# Locate the project's Data folder
data_dir = Path.cwd().parent / "Data"

# Read only the first 5 rows
df_2019 = pd.read_csv(data_dir / "2019 Data.csv", nrows=5)
df_2020 = pd.read_csv(data_dir / "2020 Data.csv", nrows=5)

print("2019 columns:", len(df_2019.columns))
print("2020 columns:", len(df_2020.columns))

print("\nSame columns:", df_2019.columns.equals(df_2020.columns))

display(df_2019.head())

2019 columns: 110
2020 columns: 110

Same columns: True


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,Unnamed: 109
0,2019,1,1,4,5,2019-01-04,OO,20304,OO,N945SW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2019,1,1,4,5,2019-01-04,OO,20304,OO,N932SW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2019,1,1,4,5,2019-01-04,OO,20304,OO,N932SW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2019,1,1,4,5,2019-01-04,OO,20304,OO,N916SW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2019,1,1,4,5,2019-01-04,OO,20304,OO,N107SY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
for i, column in enumerate(df_2019.columns, start=1):
    print(f"{i:3}. {column}")

  1. Year
  2. Quarter
  3. Month
  4. DayofMonth
  5. DayOfWeek
  6. FlightDate
  7. Reporting_Airline
  8. DOT_ID_Reporting_Airline
  9. IATA_CODE_Reporting_Airline
 10. Tail_Number
 11. Flight_Number_Reporting_Airline
 12. OriginAirportID
 13. OriginAirportSeqID
 14. OriginCityMarketID
 15. Origin
 16. OriginCityName
 17. OriginState
 18. OriginStateFips
 19. OriginStateName
 20. OriginWac
 21. DestAirportID
 22. DestAirportSeqID
 23. DestCityMarketID
 24. Dest
 25. DestCityName
 26. DestState
 27. DestStateFips
 28. DestStateName
 29. DestWac
 30. CRSDepTime
 31. DepTime
 32. DepDelay
 33. DepDelayMinutes
 34. DepDel15
 35. DepartureDelayGroups
 36. DepTimeBlk
 37. TaxiOut
 38. WheelsOff
 39. WheelsOn
 40. TaxiIn
 41. CRSArrTime
 42. ArrTime
 43. ArrDelay
 44. ArrDelayMinutes
 45. ArrDel15
 46. ArrivalDelayGroups
 47. ArrTimeBlk
 48. Cancelled
 49. CancellationCode
 50. Diverted
 51. CRSElapsedTime
 52. ActualElapsedTime
 53. AirTime
 54. Flights
 55. Distance
 56. DistanceGroup
 5

In [18]:
print("2019 data types:")
print(df_2019.dtypes.to_string())

print("\nMissing values in sample:")
print(df_2019.isna().sum().to_string())

2019 data types:
Year                                 int64
Quarter                              int64
Month                                int64
DayofMonth                           int64
DayOfWeek                            int64
FlightDate                          object
Reporting_Airline                   object
DOT_ID_Reporting_Airline             int64
IATA_CODE_Reporting_Airline         object
Tail_Number                         object
Flight_Number_Reporting_Airline      int64
OriginAirportID                      int64
OriginAirportSeqID                   int64
OriginCityMarketID                   int64
Origin                              object
OriginCityName                      object
OriginState                         object
OriginStateFips                      int64
OriginStateName                     object
OriginWac                            int64
DestAirportID                        int64
DestAirportSeqID                     int64
DestCityMarketID                     

In [19]:
# Read a larger sample for missing-value exploration
df_sample = pd.read_csv(
    data_dir / "2020 Data.csv",
    nrows=10000
)

missing_summary = pd.DataFrame({
    "Missing_Count": df_sample.isna().sum(),
    "Missing_Percentage": (
        df_sample.isna().mean() * 100
    ).round(2)
})

missing_summary.sort_values(
    "Missing_Percentage",
    ascending=False
).head(20)

,Missing_Count,Missing_Percentage
Unnamed: 109,10000,100.0
Div3Airport,10000,100.0
Div3WheelsOff,10000,100.0
Div3LongestGTime,10000,100.0
Div3TotalGTime,10000,100.0
Div3WheelsOn,10000,100.0
Div3AirportSeqID,10000,100.0
Div3AirportID,10000,100.0
Div2TailNum,10000,100.0
Div4Airport,10000,100.0


In [20]:
selected_columns = [
    "Year",
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "OriginCityName",
    "Dest",
    "DestCityName",
    "DepDelay",
    "DepDelayMinutes",
    "ArrDelay",
    "ArrDelayMinutes",
    "Cancelled",
    "CancellationCode",
    "Diverted",
    "Distance",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
]

# Check that all selected columns exist
missing_columns = [
    col for col in selected_columns
    if col not in df_2020.columns
]

print("Selected columns:", len(selected_columns))
print("Missing columns:", missing_columns)

# Preview only the columns we need
display(df_sample[selected_columns].head())

Selected columns: 24
Missing columns: []


,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Flight_Number_Reporting_Airline,Origin,OriginCityName,Dest,...,ArrDelayMinutes,Cancelled,CancellationCode,Diverted,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2020,1,17,5,2020-01-17,B6,942,PBI,"West Palm Beach/Palm Beach, FL",BDL,...,0.0,0.0,NaN,0.0,1133.0,NaN,NaN,NaN,NaN,NaN
1,2020,1,18,6,2020-01-18,B6,942,PBI,"West Palm Beach/Palm Beach, FL",BDL,...,0.0,0.0,NaN,0.0,1133.0,NaN,NaN,NaN,NaN,NaN
2,2020,1,19,7,2020-01-19,B6,942,PBI,"West Palm Beach/Palm Beach, FL",BDL,...,0.0,0.0,NaN,0.0,1133.0,NaN,NaN,NaN,NaN,NaN
3,2020,1,20,1,2020-01-20,B6,942,PBI,"West Palm Beach/Palm Beach, FL",BDL,...,4.0,0.0,NaN,0.0,1133.0,NaN,NaN,NaN,NaN,NaN
4,2020,1,21,2,2020-01-21,B6,942,PBI,"West Palm Beach/Palm Beach, FL",BDL,...,6.0,0.0,NaN,0.0,1133.0,NaN,NaN,NaN,NaN,NaN


In [21]:
# Read only the 24 columns we selected
df_2020 = pd.read_csv(
    data_dir / "2020 Data.csv",
    usecols=selected_columns,
    parse_dates=["FlightDate"],
    low_memory=False
)

print("Rows and columns:", df_2020.shape)
display(df_2020.head())

Rows and columns: (607346, 24)


,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Flight_Number_Reporting_Airline,Origin,OriginCityName,Dest,...,ArrDelayMinutes,Cancelled,CancellationCode,Diverted,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2020,1,17,5,2020-01-17,B6,942,PBI,"West Palm Beach/Palm Beach, FL",BDL,...,0.0,0.0,NaN,0.0,1133.0,NaN,NaN,NaN,NaN,NaN
1,2020,1,18,6,2020-01-18,B6,942,PBI,"West Palm Beach/Palm Beach, FL",BDL,...,0.0,0.0,NaN,0.0,1133.0,NaN,NaN,NaN,NaN,NaN
2,2020,1,19,7,2020-01-19,B6,942,PBI,"West Palm Beach/Palm Beach, FL",BDL,...,0.0,0.0,NaN,0.0,1133.0,NaN,NaN,NaN,NaN,NaN
3,2020,1,20,1,2020-01-20,B6,942,PBI,"West Palm Beach/Palm Beach, FL",BDL,...,4.0,0.0,NaN,0.0,1133.0,NaN,NaN,NaN,NaN,NaN
4,2020,1,21,2,2020-01-21,B6,942,PBI,"West Palm Beach/Palm Beach, FL",BDL,...,6.0,0.0,NaN,0.0,1133.0,NaN,NaN,NaN,NaN,NaN


In [22]:
df_2020.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 607346 entries, 0 to 607345
Data columns (total 24 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   Year                             607346 non-null  int64         
 1   Month                            607346 non-null  int64         
 2   DayofMonth                       607346 non-null  int64         
 3   DayOfWeek                        607346 non-null  int64         
 4   FlightDate                       607346 non-null  datetime64[ns]
 5   Reporting_Airline                607346 non-null  object        
 6   Flight_Number_Reporting_Airline  607346 non-null  int64         
 7   Origin                           607346 non-null  object        
 8   OriginCityName                   607346 non-null  object        
 9   Dest                             607346 non-null  object        
 10  DestCityName                     607346 non-

In [23]:
df_2020.to_sql(
    name="flights_2020",
    con=project_engine,
    if_exists="fail",
    index=False,
    chunksize=5000,
    method="multi"
)

print("2020 data imported successfully!")

2020 data imported successfully!


In [24]:
from sqlalchemy import text

with project_engine.connect() as conn:
    result = pd.read_sql(
        text("""
            SELECT
                COUNT(*) AS total_flights,
                MIN("FlightDate") AS first_date,
                MAX("FlightDate") AS last_date
            FROM flights_2020;
        """),
        conn
    )

display(result)

,total_flights,first_date,last_date
0,607346,2020-01-01,2020-01-31
